# Latihan 2 (Bonus): Endpoint HTTP Streaming (SSE)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-genai-migas/blob/main/colab/hari3/bonus-streaming-agen-latihan-2.ipynb)

Ini versi **Google Colab**. Isi dan urutan contohnya sama dengan versi VS Code; yang berbeda hanya cell Persiapan di bawah, yang menyalin repository, memasang pustaka, dan mengisi API key di runtime Colab.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kode jawaban latihan ada di bagian **Kode Jawaban Latihan** pada [PDF bab ini](https://github.com/dteti-workshop/workshop-genai-migas/blob/main/hari3/bonus-streaming-agen/bonus-streaming-agen.pdf). Salin kodenya ke cell yang bertanda `TODO`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-genai-migas.git"
REPO = "/content/workshop-genai-migas" if os.path.isdir("/content") else os.path.abspath("workshop-genai-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

Latihan ini berupa file Python, `hari3/bonus-streaming-agen/latihan_2.py`. Di Colab file itu ditulis dari cell di bawah. Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab, lalu jalankan cell untuk menyimpan file. Jalankan ulang cell setiap kali kodenya diubah.

Baris pertama cell (`%%writefile ...`) menentukan nama file dan tidak boleh diubah.

In [ ]:
%%writefile hari3/bonus-streaming-agen/latihan_2.py
"""
Bab bonus Streaming Agen - Latihan Bonus: Endpoint HTTP Streaming (SSE)

Server web kecil, hanya dengan library standar Python, yang mengalirkan jawaban agen ke klien sebagai
Server-Sent Events (SSE):

    GET /chat?q=<pertanyaan>   ->   data: {"type": "token", "text": "..."}\\n\\n   (berulang)
                                    data: {"type": "done"}\\n\\n

Browser dapat membacanya dengan API EventSource bawaan; di sini klien Python yang melakukannya.
(Di proyek nyata Anda memakai framework seperti FastAPI, tetapi gagasannya identik.)

Jalankan demo mandiri:  python hari3/bonus-streaming-agen/latihan_2.py --demo
Jalankan server saja:   python hari3/bonus-streaming-agen/latihan_2.py
                        lalu buka http://localhost:8000/chat?q=Apa%20itu%20water%20cut
"""

import json
import os
import sys
import threading
import urllib.parse
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import AIMessageChunk, HumanMessage
from langchain_openai import ChatOpenAI

load_dotenv()


def buat_agen():
    model = ChatOpenAI(
        model=os.getenv("ONLINE_MODEL"),
        base_url=os.getenv("ONLINE_BASE_URL"),
        api_key=os.getenv("ONLINE_API_KEY"),
        reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
    )
    return create_agent(
        model, tools=[], system_prompt="Anda asisten operasi produksi migas. Jawab dalam 2-3 kalimat pendek Bahasa Indonesia."
    )


AGEN = buat_agen()


def event_sse(payload: dict) -> bytes:
    """Format satu Server-Sent Event."""
    # TODO(BS.4): kembalikan satu Server-Sent Event: teks 'data: <JSON>' diakhiri dua baris baru, dalam bytes UTF-8
    raise NotImplementedError("TODO(BS.4)")


class PenanganChat(BaseHTTPRequestHandler):
    def do_GET(self):
        url = urllib.parse.urlparse(self.path)
        if url.path != "/chat":
            self.send_error(404, "Pakai /chat?q=pertanyaan+anda")
            return
        pertanyaan = urllib.parse.parse_qs(url.query).get("q", [""])[0].strip()
        if not pertanyaan:
            self.send_error(400, "Parameter ?q= belum diisi")
            return

        self.send_response(200)
        self.send_header("Content-Type", "text/event-stream")  # memberi tahu klien bahwa ini aliran SSE
        self.send_header("Cache-Control", "no-cache")
        self.end_headers()

        try:
            # TODO(BS.5): alirkan AGEN.stream(..., stream_mode="messages"); untuk setiap chunk teks dari node model tulis event token ke self.wfile lalu flush; setelah selesai tulis event done
            pass  # TODO: ganti dengan implementasi Anda
        except (BrokenPipeError, ConnectionResetError):
            pass  # klien menutup koneksi; hentikan streaming

    def log_message(self, format, *args):
        pass  # jaga konsol tetap tenang


def jalankan_klien(port: int, pertanyaan: str) -> None:
    """Klien SSE minimal: baca respons baris demi baris saat tiba."""
    url = f"http://127.0.0.1:{port}/chat?q={urllib.parse.quote(pertanyaan)}"
    print(f"Permintaan klien: GET {url}\n")
    print("Agen: ", end="", flush=True)
    jumlah = 0
    with urllib.request.urlopen(url) as response:
        for baris_mentah in response:
            baris = baris_mentah.decode("utf-8").strip()
            if not baris.startswith("data:"):
                continue  # baris pemisah kosong
            event = json.loads(baris[len("data:"):])
            jumlah += 1
            if event["type"] == "token":
                print(event["text"], end="", flush=True)
            elif event["type"] == "done":
                print(f"\n\nAliran selesai setelah {jumlah} event")


def main():
    if "--demo" in sys.argv:
        server = ThreadingHTTPServer(("127.0.0.1", 0), PenanganChat)  # port 0 = pilih port kosong
        port = server.server_address[1]
        threading.Thread(target=server.serve_forever, daemon=True).start()
        print(f"Server berjalan di http://127.0.0.1:{port}\n")
        jalankan_klien(port, "Apa bedanya alarm H2S tingkat 1 dan tingkat 2?")
        server.shutdown()
        return

    server = ThreadingHTTPServer(("127.0.0.1", 8000), PenanganChat)
    print("Mendengarkan di http://localhost:8000/chat?q=... (Ctrl+C untuk berhenti)")
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        print("\nDihentikan")


if __name__ == "__main__":
    main()

Setelah file tersimpan, jalankan demo mandiri:

In [ ]:
!{sys.executable} hari3/bonus-streaming-agen/latihan_2.py --demo